In [1]:
print("OK")

OK


In [2]:
%pwd

'/Users/rheagupta/Desktop/Medical Chatbot/Medical-Chatbot-Generative-AI/research'

In [3]:
import os
os.chdir("../")

In [4]:
%pwd

'/Users/rheagupta/Desktop/Medical Chatbot/Medical-Chatbot-Generative-AI'

In [5]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [7]:
# Extract data from PDF File
def load_pdf_file(data):
    loader = DirectoryLoader(data,
                            glob = '*.pdf',
                            loader_cls = PyPDFLoader)
    documents=loader.load()
    return documents

In [8]:
extracted_data = load_pdf_file(data = 'Data/')

In [9]:
# Split the Data into Chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap = 20)
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks


In [10]:
text_chunks = text_split(extracted_data)
print("length of text chunks", len(text_chunks))

length of text chunks 5859


In [11]:
from langchain.embeddings import HuggingFaceEmbeddings

# Download the embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings= HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
    return embeddings


In [12]:
embeddings = download_hugging_face_embeddings()

/var/folders/bb/b2rjc_f550xdy_x199g_3lb00000gn/T/ipykernel_13578/654081678.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings= HuggingFaceEmbeddings(model_name = 'sentence-transformers/all-MiniLM-L6-v2')
/Users/rheagupta/miniconda3/envs/medibot/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
query_result = embeddings.embed_query("Hello World")
print("Len", len(query_result))

Len 384


/Users/rheagupta/miniconda3/envs/medibot/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [14]:
from dotenv import load_dotenv
load_dotenv()

True

In [15]:
import os
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
GOOGLE_API_KEY = os.environ.get('GOOGLE_API_KEY')

In [16]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key = PINECONE_API_KEY)
index_name  = 'medicalbot'

pc.create_index(
    name=index_name,
    dimension = 384,
    metric="cosine",
    spec = ServerlessSpec(
        cloud="aws",
        region="us-east-1",
    )
)

{
    "name": "medicalbot",
    "metric": "cosine",
    "host": "medicalbot-vsz5yw7.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}

In [17]:
import os
os.environ["PINCONE_API_KEY"] = PINECONE_API_KEY 
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY 

In [18]:
# Embed each chunk and insert the embeddings into your pinecone index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents = text_chunks,
    index_name = index_name,
    embedding = embeddings,
)

/Users/rheagupta/miniconda3/envs/medibot/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [19]:
# Load Existing Index

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name  = index_name,
    embedding = embeddings,
)

In [20]:
docsearch


In [21]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs = {"k":3})

In [22]:
retrieved_docs = retriever.invoke("What is Acne?")

/Users/rheagupta/miniconda3/envs/medibot/lib/python3.10/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [23]:
retrieved_docs

[Document(id='9ef257cd-ab18-49c1-bbf5-279c7544ad5b', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data/Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='5ae079ec-b713-4889-a1dd-0e1bbf4da6b1', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 38.0, 'page_label': '39', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data/Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 

In [51]:
def query_pinecone(question, top_k=3):
    question_vector = embedder.encode([question])[0].tolist()  # convert to list for Pinecone

    # Query Pinecone for top_k similar vectors
    result = index.query(
        vector=question_vector,
        top_k=top_k,
        include_metadata=True  # Assuming your data chunks are stored in metadata
    )
    
    # Extract the text chunks from metadata
    retrieved_chunks = [match['metadata']['text'] for match in result['matches']]
    return retrieved_chunks

In [74]:
import google.generativeai as genai

model = genai.GenerativeModel(model_name="models/gemini-1.5-flash-latest")

def answer_with_gemini(question):
    retrieved_context = query_pinecone(question)
    context_text = "\n\n".join(retrieved_context)
    
    prompt = (
        "You are an assistant for question-answering tasks. "
        "Use the following piece of retrieved context to answer the question."
        "If the answer is not found in the context, say that you dont know the answer."
        "Use three sentences maximum and keep the answer concise."
        "\n\n"
        "{context}"
    )
    
    response = model.generate(prompt)
    return response.text

In [75]:
import os
from dotenv import load_dotenv
import google.generativeai as genai

# Load API key
load_dotenv()
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))



In [76]:
model = genai.GenerativeModel(model_name="models/gemini-1.5-flash-latest")

In [77]:
response = model.generate_content("What is Acne?")
print(response.text)

Acne is a common skin condition that occurs when hair follicles become plugged with oil and dead skin cells.  This can lead to the formation of various blemishes, including:

* **Whiteheads:** Closed pores filled with sebum (oil) and dead skin cells.
* **Blackheads:** Open pores filled with sebum and dead skin cells that have been exposed to air, causing oxidation and darkening.
* **Papules:** Small, red, inflamed bumps.
* **Pustules:** Papules with pus at the tip.
* **Nodules:** Larger, deeper, and more painful lumps under the skin.
* **Cysts:** Large, painful, pus-filled lumps that can leave scars.

Acne typically affects the face, forehead, chest, back, and shoulders.  The severity of acne varies greatly from person to person, ranging from mild (a few occasional blemishes) to severe (many inflamed lesions and potential scarring).

The exact cause of acne is not fully understood, but several factors contribute, including:

* **Hormonal changes:**  Androgens, hormones that increase du

In [78]:
response = model.generate_content("What is Gigantism?")
print(response.text)

Gigantism is a condition characterized by excessive growth and height due to an overproduction of growth hormone (GH) before the growth plates in the bones close.  This typically happens during childhood and adolescence.  Unlike acromegaly (which occurs after the growth plates close), gigantism results in disproportionately large stature affecting the entire body.  The excessive GH leads to increased cell growth and division throughout the body, resulting in significantly taller than average height, along with enlarged organs and other physical changes.

